# FAD Demo on W_030 EEG (caregiver, Brave)

This notebook mirrors the FAD workflow from the MATLAB live-script idea, using the local NetCDF EEG data and the new Python FAD implementation in src/mtmvar.py.

Signal used: Fz channel from data/UNIWAW_imported/EEG/W_030/caregiver/W_030_EEG_cg_Brave.nc.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.mtmvar import fad_decomposition, fad_components_table

In [ ]:
# Load EEG dataset
nc_path = '../data/UNIWAW_imported/EEG/W_030/caregiver/W_030_EEG_cg_Brave.nc'
ds = xr.open_dataset(nc_path)
selected_channel = 'Fz'  # Change as needed
# The file has one main data variable with a descriptive name
var_name = list(ds.data_vars)[0]
sig = ds[var_name]

# Select channel
if 'channel' not in sig.dims:
    raise ValueError("Expected channel dim, got: %s" % (sig.dims,))
if selected_channel not in ds['channel'].values:
    raise ValueError('Channel %s not found in dataset' % selected_channel)

x = sig.sel(channel=selected_channel).values.astype(float)
t = ds['time'].values.astype(float)

# Estimate sampling frequency from time coordinate
dt = float(np.median(np.diff(t)))
fs = 1.0 / dt

print('Variable:', var_name)
print('Signal shape:', x.shape)
print('Estimated fs: %.6f Hz' % fs)

In [ ]:
# Quick view of the signal
plt.figure(figsize=(12, 3))
plt.plot(t, x, lw=0.8)
plt.title(f'W_030 EEG caregiver Brave - {selected_channel} channel')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# FAD decomposition
# model_order=None => automatic order selection by AIC over max_model_order
fad = fad_decomposition(
    x,
    fs=fs,
    model_order=None,
    max_model_order=12,
    crit_type='AIC',
    plot=True,
    pair_conjugates=True
)

print('Selected AR order:', fad['model_order'])
print('Number of oscillatory components (paired):', len(fad['paired_components']['freq_hz']))

In [ ]:
# Compact FAD table (DataFrame)
fad_df = fad_components_table(fad, output='dataframe', decimals=6)
fad_df

In [ ]:
# Optional export for downstream analysis
#out_csv = 'fad_w030_fz_components.csv'
#fad_df.to_csv(out_csv, index=False)
#print('Saved:', out_csv)

## Compare with specparam

In [ ]:
# Import the specparam spectral model
from specparam import SpectralModel